In [ ]:
# epoch影响较大
%run ../stage1.py \
    --sc_file "/mnt/d/ST_Graduation_Project_data/database/GSE144236/GSE144236_P2_SC.h5ad" \
    --st_file "/mnt/d/ST_Graduation_Project_data/database/GSE144236/GSE144236_P2_ST.h5ad" \
    --n_epochs 150 \
    --output_dir ../deconv_results/GSE144236_P2
 #   --marker_selection_method l1 
 #   --precomputed_marker_file "/mnt/d/ST_Graduation_Project/SC_MAP_ST/deconv_results/GSE243275/final_genes.txt"

/home/mwc/miniconda3/envs/dl/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu
Stage 1 Training: VAE (SC + ST, Marker Genes)
Configuration:
   Marker genes per type: 100
   Leiden resolution: 4
   Batch size: 1024
   Epochs: 150
   Learning rate: 0.0005
   Beta (KL weight): 0.1
   Hidden dims: [512, 256]
   Latent dim: 128
   Loss type: MSE
   Lambda MMD: 0.05
   Dual Decoder: True
   Aggregation method: mean
Loading datasets...
   Loading SC: /mnt/d/ST_Graduation_Project_data/database/GSE144236/GSE144236_P2_SC.h5ad
   SC shape: (6824, 24542)
   SC avg counts/cell: 10607.7 (from X)
   Loading ST: /mnt/d/ST_Graduation_Project_data/database/GSE144236/GSE144236_P2_ST.h5ad
   ST shape: (646, 17344)
   Common genes: 16278
   SC final: (6824, 16278)
   ST final: (646, 16278)
Computing clusters and marker genes...
Starting clustering analysis...
Clustering results: 63 clusters
Marker genes per cluster:
   0: 100 -> 100 (after L1)
   1: 100 -> 100 (after L1)
   10: 100 -> 100 (after L1)
   11: 100 -> 100 (after L1)
   12: 100 -> 50 (after L1)
   13: 100

VAE Training: 100%|██████████| 150/150 [01:09<00:00,  2.15epoch/s, Train=316.4105, Recon=310.0394, KL=63.7114, MMD=0.0120, Test=304.4245] 


📊 Computing cluster centers using ALL SC data (train + test)...
   ALL SC data: (6824, 1475)
   Number of clusters: 63
   Computed embeddings shape: (6824, 128)
Computing cluster centers with mean aggregation...
   Completed: 63 clusters with mean centers and expressions
Extracting celltype-cluster mapping (using 'cell_type' column)...
Visualizing modality alignment...
Generating UMAP visualization for modality alignment...
   Computing UMAP on 7394 samples with 128 dims...
   UMAP visualization saved to: ../deconv_results/GSE144236_P2/modality_alignment_umap.png
Saving model to: ../deconv_results/GSE144236_P2/final_vae.pth
   Average cell counts: 10607.7 (for Stage 2 scale factor)
✅ Saved VAE model (weights only): ../deconv_results/GSE144236_P2/final_vae.pth
Saving cluster data to: ../deconv_results/GSE144236_P2/final_vae_cluster_data.npz
   ✓ Cluster IDs: 63
   ✓ Prototypes: (63, 128)
   ✓ Expressions (marker): (63, 1475)
   ✓ Expressions (full): 63 clusters × 16278 genes
   ✓ Cellty

In [2]:
%run ../stage2.py \
    --st_file "/mnt/d/ST_Graduation_Project_data/database/GSE144236/GSE144236_P2_ST.h5ad" \
    --stage1_model_path "../deconv_results/GSE144236_P2/final_vae.pth" \
    --output_dir "../deconv_results/GSE144236_P2" \
    --n_epochs 250

Sample name: GSE144236_P2
Stage 1 model: ../deconv_results/GSE144236_P2/final_vae.pth
Output directory: ../deconv_results/GSE144236_P2
Device: cpu
Weight threshold: 0.001
Loading pretrained VAE Encoder...
   VAE architecture: 1475 -> 128
   Output type: mse
   Architecture: Dual Decoder (SC/ST-specific)
   ✓ Loaded 6824 cell cluster labels from checkpoint
Loading cluster data from: ../deconv_results/GSE144236_P2/final_vae_cluster_data.npz
   Cluster prototypes: torch.Size([63, 128])
   Cluster expressions (marker): torch.Size([63, 1475])
   Cluster expressions (all genes): 63 × 16278
   Loaded celltype mapping: 63 clusters
   Average cell counts: 10607.7
Loaded all genes list: 16278 genes
VAE Encoder loaded: 1475 -> 128
Cell type clusters: ['0', '1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '3', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '4', '40', '41', '42', '43', '44', '45', '46', '4

GAT Training: 100%|██████████| 250/250 [06:41<00:00,  1.61s/epoch, Total=3.2020, MSE=0.3526, Spot_Cosine=0.1744, Gene_Cosine=0.2460, Pearson=0.2552, Gene_Pearson=0.3449, P_pat=37, M_pat=37, C_pat=34]


Evaluating model results...
Applying weight threshold: 0.001
   Non-zero elements: 19380 -> 11166 (27.4%)
Saving deconvolution results...
Generating deconvolution expression matrices...
   Reconstructing all gene expression...
   ✅ Scale factor computed from MARKER genes (1475 genes)
      Spot marker total (real): mean=6075.8
      Reconstructed marker total: mean=1066.4
      Scale factor: min=0.0507, max=37.5256, mean=5.5907
   Saved: ../deconv_results/GSE144236_P2/GSE144236_P2_reconstructed_all_genes.csv
   Cell type composition...
   Using cluster→celltype mapping from Stage 1 checkpoint (63 entries).
   Found duplicate celltype names: ['CD1C', 'Tcell', 'Epithelial', 'Melanocyte', 'ASDC', 'Endothelial Cell', 'LC', 'Mac']. Merging corresponding cluster columns by summing weights.
   Columns before: 63, after merge: 12
   Saved cell composition (celltype): ../deconv_results/GSE144236_P2/GSE144236_P2_cell_composition.csv
   Saved cluster composition: ../deconv_results/GSE144236_P2/GS